# Static Baselines: Paper Reproduction + Practitioner Analytics

**next-gen-aiam** · `notebooks/01_static_baselines.ipynb`

This notebook has two parts:

1. **Paper reproduction** — every table and figure in the paper, computed from cached parquets, with a reproducibility-assertion cell at the end.
2. **Practitioner analytics** — rolling Sharpe, calendar-year heatmap, drawdown, CVaR, turnover, weight distribution, allocation pie/timeline/snapshots, strategy correlation matrix, bootstrap CIs, and pairwise Sharpe tests.


In [ ]:
import os, sys, warnings
from pathlib import Path

# Work from project root regardless of where Jupyter was launched
ROOT = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path.cwd()
while ROOT.name != 'next-gen-aiam' and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
warnings.filterwarnings('ignore')
print('Working dir:', Path.cwd())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mtick
from matplotlib.patches import FancyBboxPatch
from matplotlib.ticker import FuncFormatter
import matplotlib.patheffects as pe
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import squareform

from aiam.evaluation.transaction_costs import apply_costs, compute_turnover
from aiam.harness.horse_race import _weights_path

# ── Style ─────────────────────────────────────────────────────────────────────
matplotlib.rcParams.update({
    "font.family":        "serif",
    "font.size":          10,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.alpha":         0.20,
    "grid.linestyle":     "--",
    "figure.facecolor":   "white",
    "axes.facecolor":     "white",
    "axes.labelsize":     10,
    "legend.fontsize":    8,
    "xtick.labelsize":    8,
    "ytick.labelsize":    8,
})

FAMILY_COLORS = {
    "Classical MV":    "#1f77b4",
    "Diversification": "#2ca02c",
    "Regime":          "#9467bd",
    "TSMOM":           "#ff7f0e",
    "Black-Litterman": "#d62728",
    "FF3 Factor":      "#8c564b",
    "EW (benchmark)":  "#333333",
}

FAMILY_MAP = {
    "EW":               "EW (benchmark)",
    "GMV(sample)":      "Classical MV",
    "GMV(ledoit_wolf)": "Classical MV",
    "GMV(oas)":         "Classical MV",
    "MSR(sample)":      "Classical MV",
    "MSR(ledoit_wolf)": "Classical MV",
    "MDP(sample)":      "Diversification",
    "MDP(ledoit_wolf)": "Diversification",
    "RP(sample)":       "Diversification",
    "RP(ledoit_wolf)":  "Diversification",
    "HRP(sample)":      "Diversification",
    "HRP(ledoit_wolf)": "Diversification",
    "SWITCH(sample)":   "Regime",
    "SWITCH(ledoit_wolf)": "Regime",
    "TSMOM(12m)":       "TSMOM",
    "TSMOM(6m)":        "TSMOM",
    "BL-Eq(sample)":    "Black-Litterman",
    "BL-Eq(LW)":        "Black-Litterman",
    "BL-Mom(LW)":       "Black-Litterman",
    "BL-Rev(LW)":       "Black-Litterman",
    "FF3-Mom":          "FF3 Factor",
    "FF3-LowVol":       "FF3 Factor",
    "FF3-Quality":      "FF3 Factor",
    "FF3-Multi":        "FF3 Factor",
}

DISPLAY = {
    "EW":                "EW",
    "GMV(sample)":       "GMV(sample)",
    "GMV(ledoit_wolf)":  "GMV(LW)",
    "GMV(oas)":          "GMV(OAS)",
    "MSR(sample)":       "MSR(sample)",
    "MSR(ledoit_wolf)":  "MSR(LW)",
    "MDP(sample)":       "MDP(sample)",
    "MDP(ledoit_wolf)":  "MDP(LW)",
    "RP(sample)":        "RP(sample)",
    "RP(ledoit_wolf)":   "RP(LW)",
    "HRP(sample)":       "HRP(sample)",
    "HRP(ledoit_wolf)":  "HRP(LW)",
    "SWITCH(sample)":    "SWITCH(sample)",
    "SWITCH(ledoit_wolf)": "SWITCH(LW)",
    "TSMOM(12m)":        "TSMOM(12m)",
    "TSMOM(6m)":         "TSMOM(6m)",
    "BL-Eq(sample)":     "BL-Eq(sample)",
    "BL-Eq(LW)":         "BL-Eq(LW)",
    "BL-Mom(LW)":        "BL-Mom(LW)",
    "BL-Rev(LW)":        "BL-Rev(LW)",
    "FF3-Mom":           "FF3-Mom",
    "FF3-LowVol":        "FF3-LowVol",
    "FF3-Quality":       "FF3-Quality",
    "FF3-Multi":         "FF3-Multi",
}

CRISES = [
    (pd.Timestamp("2008-09-01"), pd.Timestamp("2009-03-31"), "GFC"),
    (pd.Timestamp("2020-02-01"), pd.Timestamp("2020-04-30"), "COVID"),
    (pd.Timestamp("2022-01-01"), pd.Timestamp("2022-10-31"), "Rate Shock"),
]

COST_BPS = 10.0
TRADING_DAYS = 252


In [ ]:
from aiam.evaluation.switch_assembly import assemble_switch_returns
from aiam.evaluation.vmp_assembly import assemble_vmp_returns
from aiam.evaluation.regime_conditional import regime_conditional_performance

WEIGHTS_SUFFIX = "29assets_2003_2026"
BASE_24_COLS = [
    "EW", "GMV(sample)", "GMV(ledoit_wolf)", "GMV(oas)", "MSR(sample)", "MSR(ledoit_wolf)",
    "MDP(sample)", "MDP(ledoit_wolf)", "RP(sample)", "RP(ledoit_wolf)", "HRP(sample)", "HRP(ledoit_wolf)",
    "SWITCH(sample)", "SWITCH(ledoit_wolf)", "TSMOM(12m)", "TSMOM(6m)",
    "BL-Eq(sample)", "BL-Eq(LW)", "BL-Mom(LW)", "BL-Rev(LW)",
    "FF3-Mom", "FF3-LowVol", "FF3-Quality", "FF3-Multi",
]

all_base = pd.read_parquet("data/cache/portfolio_returns/31strategies_29assets_2003_2026.parquet")
all_vmp  = pd.read_parquet("data/cache/portfolio_returns/31strategies_vmp_29assets_2003_2026.parquet")
base = all_base[BASE_24_COLS]
vmp  = all_vmp[[f"VMP({c})" for c in BASE_24_COLS]]
regime_sig = pd.read_parquet("data/cache/regime_signals_2003_2026.parquet")["dominant_regime"].dropna()
prices = pd.read_parquet("data/cache/prices_29.parquet")

# Compute regime-conditional Sharpe (12 baseline strategies x 8 regimes) on the fly
HEATMAP_COLS = [
    "EW", "GMV(sample)", "GMV(ledoit_wolf)", "GMV(oas)", "MSR(sample)", "MSR(ledoit_wolf)",
    "MDP(sample)", "MDP(ledoit_wolf)", "RP(sample)", "RP(ledoit_wolf)", "HRP(sample)", "HRP(ledoit_wolf)",
]
reg_tables = regime_conditional_performance({s: base[s].dropna() for s in HEATMAP_COLS}, regime_sig)
regime_conditional_sharpe = reg_tables["sharpe"]

# Forward-fill regime to daily (for heatmap nday counts)
regime_daily = regime_sig.resample("B").ffill().reindex(base.index, method="ffill")

# Build SWITCH(v2a) using assemble_switch_returns
SWITCH_V2A_RULE = {0: "MSR(ledoit_wolf)", 5: "MSR(sample)"}
switch_v2a = assemble_switch_returns(
    base, regime_sig, SWITCH_V2A_RULE, "MDP(ledoit_wolf)"
).rename("SWITCH(v2a)")

# Build VMP(SWITCH(v2a))
target_vol = switch_v2a.std() * np.sqrt(TRADING_DAYS)
roll_vol   = switch_v2a.rolling(21).std() * np.sqrt(TRADING_DAYS)
exposure   = (target_vol / roll_vol.shift(1)).clip(0.25, 1.5)
vmp_switch_v2a = (switch_v2a * exposure).rename("VMP(SWITCH(v2a))")

# SPY daily returns
spy_ret = prices["SPY.US"].pct_change().dropna()

# Load weight DataFrames (29-asset suffix)
weights_cache = {}
for col in BASE_24_COLS:
    p = _weights_path(col, suffix=WEIGHTS_SUFFIX)
    if p.exists():
        weights_cache[col] = pd.read_parquet(p)

print(f"base: {base.shape}, vmp: {vmp.shape}")
print(f"regime_daily: {regime_daily.shape}, prices: {prices.shape}")
print(f"SWITCH(v2a) sharpe: {switch_v2a.mean()/switch_v2a.std()*np.sqrt(252):.3f}")
print(f"VMP(SWITCH(v2a)) sharpe: {vmp_switch_v2a.mean()/vmp_switch_v2a.std()*np.sqrt(252):.3f}")
print(f"Weights loaded: {len(weights_cache)}/{len(BASE_24_COLS)}")


In [ ]:
def ann_sharpe(s):
    return s.mean() / s.std() * np.sqrt(TRADING_DAYS)

def ann_return(s):
    return (1 + s).prod() ** (TRADING_DAYS / len(s)) - 1

def ann_vol(s):
    return s.std() * np.sqrt(TRADING_DAYS)

def max_dd(s):
    c = (1 + s).cumprod()
    return ((c - c.cummax()) / c.cummax()).min()

def calmar(s):
    dd = max_dd(s)
    return ann_return(s) / abs(dd) if dd != 0 else np.nan

def cum_wealth(s):
    return (1 + s).cumprod()

def shade_crises(ax, alpha=0.10):
    for start, end, _ in CRISES:
        ax.axvspan(start, end, color="grey", alpha=alpha, zorder=0)

def net_sharpe_for(col, cost_bps=10.0):
    rets = base[col] if col in base.columns else vmp[col]
    # Use base weights for both base and VMP variants
    base_col = col.replace("VMP(", "").rstrip(")") if col.startswith("VMP(") else col
    w = weights_cache.get(base_col)
    if w is None:
        return np.nan
    net = apply_costs(rets, w, cost_bps=cost_bps)
    return ann_sharpe(net)

def turnover_for(col):
    base_col = col.replace("VMP(", "").rstrip(")") if col.startswith("VMP(") else col
    w = weights_cache.get(base_col)
    if w is None:
        return np.nan
    return compute_turnover(w).dropna().mean() * 100

def strategy_returns(col):
    """Fetch returns for any column name (base, VMP, or special strategies)."""
    if col == "SWITCH(v2a)":
        return switch_v2a
    if col == "VMP(SWITCH(v2a))":
        return vmp_switch_v2a
    if col in base.columns:
        return base[col]
    if col in vmp.columns:
        return vmp[col]
    raise KeyError(f"Unknown strategy: {col}")

# Precompute full stats for all 48 base + vmp strategies
ALL_STRATS = []
for col in base.columns:
    s = base[col]
    ALL_STRATS.append({
        "col": col, "display": DISPLAY[col], "is_vmp": False,
        "family": FAMILY_MAP.get(col, "EW (benchmark)"),
        "ann_ret": ann_return(s)*100, "ann_vol": ann_vol(s)*100,
        "sharpe": ann_sharpe(s), "max_dd": max_dd(s)*100,
        "calmar": calmar(s), "turnover": turnover_for(col),
        "net_sharpe": net_sharpe_for(col),
    })
for col in vmp.columns:
    s = vmp[col]
    base_col = col.replace("VMP(", "").rstrip(")")
    ALL_STRATS.append({
        "col": col, "display": col.replace("ledoit_wolf", "LW").replace("(oas)", "(OAS)"),
        "is_vmp": True,
        "family": FAMILY_MAP.get(base_col, "EW (benchmark)"),
        "ann_ret": ann_return(s)*100, "ann_vol": ann_vol(s)*100,
        "sharpe": ann_sharpe(s), "max_dd": max_dd(s)*100,
        "calmar": calmar(s), "turnover": turnover_for(col),
        "net_sharpe": net_sharpe_for(col),
    })

df_all = pd.DataFrame(ALL_STRATS)
df_base = df_all[~df_all.is_vmp].copy()
df_vmp  = df_all[df_all.is_vmp].copy()
print(f"Total strategies: {len(df_all)}  (base={len(df_base)}, vmp={len(df_vmp)})")

def hit_ratio(s):
    """Fraction of calendar months with positive return."""
    monthly = (1 + s).resample("ME").prod() - 1
    return float((monthly > 0).sum() / len(monthly)) if len(monthly) > 0 else float("nan")


---
## Part 1 — Paper Reproduction

All tables and figures from the paper, computed from cache.

### Appendix A — Full 48-Strategy Comparison Table

*Paper reference: Appendix A.*
Families: Classical MV, Diversification, Regime, TSMOM, Black-Litterman, FF3 Factor.
Each base strategy followed by its VMP variant. Numbers should match the paper exactly.


In [ ]:
FAMILY_ORDER = [
    ("Classical MV",    ["EW", "GMV(sample)", "GMV(ledoit_wolf)", "GMV(oas)", "MSR(sample)", "MSR(ledoit_wolf)"]),
    ("Diversification", ["MDP(sample)", "MDP(ledoit_wolf)", "RP(sample)", "RP(ledoit_wolf)", "HRP(sample)", "HRP(ledoit_wolf)"]),
    ("Regime",          ["SWITCH(sample)", "SWITCH(ledoit_wolf)"]),
    ("TSMOM",           ["TSMOM(12m)", "TSMOM(6m)"]),
    ("Black-Litterman", ["BL-Eq(sample)", "BL-Eq(LW)", "BL-Mom(LW)", "BL-Rev(LW)"]),
    ("FF3 Factor",      ["FF3-Mom", "FF3-LowVol", "FF3-Quality", "FF3-Multi"]),
]

def hit_ratio(s):
    monthly = (1 + s).resample("ME").prod() - 1
    return float((monthly > 0).sum() / len(monthly)) if len(monthly) > 0 else float("nan")

rows = []
for fam, cols in FAMILY_ORDER:
    for col in cols:
        s = base[col]
        vmp_col = f"VMP({col})"
        sv = vmp[vmp_col]
        for is_v, series, cname in [(False, s, col), (True, sv, vmp_col)]:
            disp = cname.replace("ledoit_wolf", "LW").replace("(oas)", "(OAS)")
            rows.append({
                "Family": fam if not is_v else "",
                "Strategy": disp,
                "Ann Ret": f"{ann_return(series)*100:.2f}%",
                "Ann Vol": f"{ann_vol(series)*100:.2f}%",
                "Sharpe":  f"{ann_sharpe(series):.3f}",
                "Hit %":   f"{hit_ratio(series)*100:.1f}%" if not is_v else "",
                "Max DD":  f"{max_dd(series)*100:.2f}%",
                "Calmar":  f"{calmar(series):.3f}",
                "Turnover": f"{turnover_for(cname):.2f}%",
                "Net Sharpe": f"{net_sharpe_for(cname):.3f}",
            })

master_table = pd.DataFrame(rows)
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_colwidth", 30)
display(master_table.to_string(index=False))


### §3.1a — Hit Ratio by Base Strategy

*Paper reference: Appendix A (Hit Ratio column).*
Fraction of calendar months with positive return, 24 base strategies sorted by hit ratio.


In [ ]:
hit_rows = []
for col in base.columns:
    s = base[col]
    hit_rows.append({
        "Strategy": DISPLAY[col],
        "Hit Ratio": hit_ratio(s),
        "Sharpe": ann_sharpe(s),
        "family": FAMILY_MAP.get(col, "EW (benchmark)"),
    })
df_hit = pd.DataFrame(hit_rows).sort_values("Hit Ratio", ascending=False)

fig, ax = plt.subplots(figsize=(12, 7))
colors_hit = [FAMILY_COLORS[r] for r in df_hit["family"]]
bars = ax.barh(df_hit["Strategy"], df_hit["Hit Ratio"] * 100, color=colors_hit, edgecolor="white", lw=0.3)
ax.axvline(50.0, color="black", ls="--", lw=0.8, alpha=0.5)
ax.text(50.2, len(df_hit) * 0.5, "50%", fontsize=7.5, color="dimgray", va="center")
ax.set_xlabel("Hit Ratio (% months positive)")
ax.set_title("Hit Ratio — 24 Base Strategies (sorted descending)", fontsize=11, fontweight="bold")
legend_patches = [mpatches.Patch(color=FAMILY_COLORS[f], label=f) for f in FAMILY_COLORS]
ax.legend(handles=legend_patches, ncol=2, fontsize=7.5, loc="lower right")
fig.tight_layout()
plt.show()

print(f"\nHit ratio range: {df_hit['Hit Ratio'].min()*100:.1f}% – {df_hit['Hit Ratio'].max()*100:.1f}%")
print(f"Median: {df_hit['Hit Ratio'].median()*100:.1f}%")


### §3.1b — Sub-Period Sharpe (5-Year Buckets)

*Paper reference: Appendix B.*
5-year sub-period Sharpe ratios for 6 representative strategies.


In [ ]:
SUBPERIODS = [
    ("2003-01-01", "2007-12-31", "2003-2007\n(Pre-GFC)"),
    ("2008-01-01", "2012-12-31", "2008-2012\n(GFC+Recovery)"),
    ("2013-01-01", "2017-12-31", "2013-2017\n(Bull Market)"),
    ("2018-01-01", "2022-12-31", "2018-2022\n(COVID+RateShock)"),
    ("2023-01-01", "2026-04-30", "2023-2026\n(AI/Rate cut)"),
]

SP_STRATS = [
    ("EW",                    "EW",             FAMILY_COLORS["EW (benchmark)"]),
    ("MSR(ledoit_wolf)",       "MSR(LW)",        FAMILY_COLORS["Classical MV"]),
    ("MSR(sample)",            "MSR(sample)",    FAMILY_COLORS["Classical MV"]),
    ("MDP(ledoit_wolf)",       "MDP(LW)",        FAMILY_COLORS["Diversification"]),
    ("SWITCH(ledoit_wolf)",    "SWITCH(LW)",     FAMILY_COLORS["Regime"]),
    (None,                     "SWITCH(v2a)",    FAMILY_COLORS["Regime"]),
    ("VMP(MSR(ledoit_wolf))",  "VMP(MSR(LW))",   FAMILY_COLORS["Classical MV"]),
    ("VMP(MDP(ledoit_wolf))",  "VMP(MDP(LW))",   FAMILY_COLORS["Diversification"]),
]

sp_rows = []
for col, label, color in SP_STRATS:
    s = switch_v2a if col is None else (vmp[col] if col in vmp.columns else base[col])
    row = {"Strategy": label}
    for start, end, period in SUBPERIODS:
        sub = s[(s.index >= start) & (s.index <= end)].dropna()
        sh = ann_sharpe(sub) if len(sub) > 30 else float("nan")
        row[period.replace("\\n", "\n")] = f"{sh:.2f}" if not pd.isna(sh) else "—"
    sp_rows.append(row)

df_sp = pd.DataFrame(sp_rows).set_index("Strategy")
print("Sub-Period Annualized Sharpe Ratios")
print("=" * 70)
display(df_sp)

# Heatmap
fig, ax = plt.subplots(figsize=(11, 6))
Z_sp = pd.DataFrame(sp_rows).set_index("Strategy").replace("—", float("nan")).astype(float)
Z_sp.columns = [c.split("\n")[0] for c in df_sp.columns]
vmax_sp = max(abs(np.nanmin(Z_sp.values)), abs(np.nanmax(Z_sp.values)))
im = ax.imshow(Z_sp.values, cmap="RdBu", vmin=-vmax_sp, vmax=vmax_sp, aspect="auto")
plt.colorbar(im, ax=ax, label="Annualized Sharpe", shrink=0.8)
for i in range(Z_sp.shape[0]):
    for j in range(Z_sp.shape[1]):
        val = Z_sp.values[i, j]
        txt = "—" if pd.isna(val) else f"{val:.2f}"
        col_txt = "white" if (not pd.isna(val) and abs(val) > vmax_sp * 0.55) else "black"
        ax.text(j, i, txt, ha="center", va="center", fontsize=9, color=col_txt, fontweight="bold")
ax.set_xticks(range(len(Z_sp.columns))); ax.set_xticklabels(Z_sp.columns, fontsize=9)
ax.set_yticks(range(len(Z_sp.index))); ax.set_yticklabels(Z_sp.index, fontsize=8.5)
ax.set_title("Sub-Period Annualized Sharpe — 5-Year Buckets", fontsize=11, fontweight="bold")
fig.tight_layout()
plt.show()


### §3.2 — Rankings

*Paper reference: §3.2 Results → Rankings.*


In [ ]:
print("── Top 10 by Sharpe (all 48 strategies) ──")
top10_sharpe = df_all.nlargest(10, "sharpe")[["display","sharpe","ann_ret","max_dd"]].reset_index(drop=True)
top10_sharpe.index += 1
display(top10_sharpe.rename(columns={"display":"Strategy","ann_ret":"Ann Ret (%)","max_dd":"Max DD (%)"}))

print()
print("── Top 5 by Annualized Return ──")
top5_ret = df_all.nlargest(5, "ann_ret")[["display","ann_ret","sharpe"]].reset_index(drop=True)
top5_ret.index += 1
display(top5_ret.rename(columns={"display":"Strategy","ann_ret":"Ann Ret (%)"}))

print()
print("── Bottom 5 by Sharpe (base strategies only) ──")
bot5 = df_base.nsmallest(5, "sharpe")[["display","sharpe","ann_ret"]].reset_index(drop=True)
bot5.index += 1
display(bot5.rename(columns={"display":"Strategy","ann_ret":"Ann Ret (%)"}))


### §3.3 — Transaction-Cost Sensitivity

*Paper reference: §3.3 Transaction-Cost Sensitivity.*
Uniform 10 bps round-trip cost applied per unit of one-way turnover.


In [ ]:
print("── Top 10 by Net Sharpe (10 bps) ──")
top10_net = df_all.nlargest(10, "net_sharpe")[["display","sharpe","net_sharpe","turnover"]].reset_index(drop=True)
top10_net.index += 1
display(top10_net.rename(columns={"display":"Strategy","sharpe":"Gross Sharpe","net_sharpe":"Net Sharpe","turnover":"Turnover (%)"}))

print()
print("── Top 5 by Sharpe Degradation (base strategies only) ──")
df_base2 = df_base.copy()
df_base2["degradation"] = df_base2["sharpe"] - df_base2["net_sharpe"]
top5_deg = df_base2.nlargest(5, "degradation")[["display","sharpe","net_sharpe","turnover","degradation"]].reset_index(drop=True)
top5_deg.index += 1
display(top5_deg.rename(columns={"display":"Strategy","sharpe":"Gross Sharpe","net_sharpe":"Net Sharpe","turnover":"Turnover (%)","degradation":"Degradation"}))


### Figure 1 — Cumulative Wealth

*Paper reference: §3.1, Figure 1 caption.*
Log-y axis. Crisis shading: GFC, COVID, Rate Shock.


In [ ]:
F1_STRATS = [
    ("EW",                    "EW (benchmark)",     FAMILY_COLORS["EW (benchmark)"],  dict(lw=2.4, ls="--", zorder=5)),
    ("VMP(MSR(ledoit_wolf))", "VMP(MSR(LW))",        FAMILY_COLORS["Classical MV"],    dict(lw=1.8, zorder=4)),
    ("VMP(BL-Mom(LW))",       "VMP(BL-Mom(LW))",    FAMILY_COLORS["Black-Litterman"], dict(lw=1.8, zorder=4)),
    (None,                    "SWITCH(v2a)",          FAMILY_COLORS["Regime"],          dict(lw=1.6, zorder=3)),
    ("BL-Mom(LW)",            "BL-Mom(LW)",          FAMILY_COLORS["Black-Litterman"], dict(lw=1.2, ls=":", zorder=3)),
    ("VMP(GMV(sample))",      "VMP(GMV(sample))",   FAMILY_COLORS["Classical MV"],    dict(lw=1.2, ls="-.", zorder=2)),
    ("FF3-LowVol",            "FF3-LowVol",          FAMILY_COLORS["FF3 Factor"],      dict(lw=1.5, zorder=3)),
]

fig, ax = plt.subplots(figsize=(12, 6))
for col, label, color, kw in F1_STRATS:
    rets = switch_v2a if col is None else (
        base[col] if col in base.columns else vmp[col]
    )
    cw = cum_wealth(rets)
    ax.semilogy(cw.index, cw.values, color=color, label=label, **kw)

shade_crises(ax)
fig.canvas.draw()
ylim = ax.get_ylim()
y_label = np.exp(np.log(ylim[0]) + 0.015 * (np.log(ylim[1]) - np.log(ylim[0])))
for start, end, label in CRISES:
    ax.text(start + (end - start)/2, y_label, label, fontsize=6.5, ha="center", va="bottom",
            color="dimgray", style="italic")

# BL-Mom(LW) trough annotation
bl = base["BL-Mom(LW)"]
cum_bl = cum_wealth(bl)
dd_bl  = (cum_bl - cum_bl.cummax()) / cum_bl.cummax()
trough_date = dd_bl.idxmin()
ax.annotate("BL-Mom(LW)\n−50.85% max DD",
    xy=(trough_date, cum_bl[trough_date]),
    xytext=(trough_date + pd.Timedelta(days=300), cum_bl[trough_date] * 0.62),
    fontsize=7.5, color=FAMILY_COLORS["Black-Litterman"],
    arrowprops=dict(arrowstyle="-", color=FAMILY_COLORS["Black-Litterman"], lw=0.8), ha="left")

ax.set_xlabel("Date")
ax.set_ylabel("Cumulative wealth (log scale, 1 = initial $1)")
ax.set_title("Cumulative Wealth, January 2008 – May 2026", fontsize=11, fontweight="bold")
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.2g}×"))
ax.legend(ncol=2, loc="upper left", framealpha=0.85)
fig.tight_layout()
plt.show()


### Figure 2 — Sharpe vs Maximum Drawdown Scatter

*Paper reference: §3.2 Rankings, Figure 2 caption.*
Filled circles = base strategies; open rings = VMP variants. Color = family.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

for family, grp in df_all.groupby("family"):
    color = FAMILY_COLORS[family]
    base_pts = grp[~grp.is_vmp]
    vmp_pts  = grp[grp.is_vmp]
    if len(base_pts):
        ax.scatter(base_pts.max_dd, base_pts.sharpe, color=color, marker="o",
                   s=50, zorder=4, edgecolors=color, label=f"{family} (base)")
    if len(vmp_pts):
        ax.scatter(vmp_pts.max_dd, vmp_pts.sharpe, color="none", marker="o",
                   s=70, zorder=4, edgecolors=color, linewidths=1.6, label=f"{family} (VMP)")

ax.axhline(1.0, color="black", ls="--", lw=0.8, alpha=0.6)
ax.axvline(-20.0, color="black", ls="--", lw=0.8, alpha=0.6)
ax.text(-19.5, df_all.sharpe.min() + 0.02, "Max DD = −20%", fontsize=7, color="dimgray", va="bottom")
ax.text(df_all.max_dd.max() * 0.5, 1.01, "Sharpe = 1.0", fontsize=7, color="dimgray", va="bottom", ha="center")

# Annotate top-5 by Sharpe
top5_pts = df_all.nlargest(5, "sharpe")
offsets = {"VMP(GMV(sample))": (3, 0.02), "VMP(MDP(sample))": (2, 0.02),
           "VMP(SWITCH(sample))": (-8, 0.04), "VMP(SWITCH(LW))": (-6, -0.06),
           "VMP(MDP(LW))": (2, -0.05)}
for _, row in top5_pts.iterrows():
    dx, dy = offsets.get(row.display, (2, 0.02))
    ax.annotate(row.display, xy=(row.max_dd, row.sharpe),
                xytext=(row.max_dd + dx, row.sharpe + dy), fontsize=7,
                arrowprops=dict(arrowstyle="-", lw=0.7, color="gray"),
                ha="left" if dx >= 0 else "right")

family_patches = [mpatches.Patch(color=FAMILY_COLORS[f], label=f) for f in FAMILY_COLORS]
base_mkr = plt.Line2D([0],[0], marker="o", color="gray", ls="", markersize=6,
                       markerfacecolor="gray", label="Base strategy")
vmp_mkr  = plt.Line2D([0],[0], marker="o", color="gray", ls="", markersize=8,
                       markerfacecolor="none", markeredgewidth=1.6, label="VMP variant")
ax.legend(handles=family_patches + [base_mkr, vmp_mkr], ncol=2, fontsize=7.5,
          loc="lower right", framealpha=0.85)
ax.set_xlabel("Maximum Drawdown (%)")
ax.set_ylabel("Sharpe Ratio (annualized)")
ax.set_title("Sharpe vs. Maximum Drawdown — All 48 Strategies", fontsize=11, fontweight="bold")
fig.tight_layout()
plt.show()


### Figure 3 — Regime-Conditional Sharpe Heatmap

*Paper reference: Finding 5 (§3 Findings), Figure 3 caption.*
12 base strategies × 8 regimes. Diverging red–blue. Gold borders = SWITCH(v2a) selection cells.


In [ ]:
REGIME_LABELS = {
    0: "R0\nExpansion", 1: "R1\nRecovery", 2: "R2\nNeutral",
    3: "R3\nSlow\nGrowth", 4: "R4\nStress",
    5: "R5\nLow &\nContracting", 6: "R6\nCrisis", 7: "R7\nContraction",
}
SPARSE_THRESHOLD = 252
HEATMAP_STRATS = list(regime_conditional_sharpe.index)

ndays = {}
for k in range(8):
    mask = (regime_daily == float(k)).reindex(base.index, fill_value=False)
    ndays[k] = mask.sum()

reg_order = [0, 1, 2, 3, 4, 5, 6, 7]
Z = regime_conditional_sharpe[reg_order].values   # shape: 12x8

vmax = min(max(abs(np.nanmin(Z)), abs(np.nanmax(Z))), 3.0)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(Z, cmap="RdBu", vmin=-vmax, vmax=vmax, aspect="auto")
plt.colorbar(im, ax=ax, label="Annualized Sharpe", shrink=0.8)

for i, strat in enumerate(HEATMAP_STRATS):
    for j, k in enumerate(reg_order):
        val = Z[i, j]
        n   = ndays[k]
        txt = "—" if np.isnan(val) else (f"{val:.2f}*" if n < SPARSE_THRESHOLD else f"{val:.2f}")
        color = "white" if (not np.isnan(val) and abs(val) > vmax * 0.6) else ("dimgray" if np.isnan(val) else "black")
        ax.text(j, i, txt, ha="center", va="center", fontsize=7, color=color)
        if n < SPARSE_THRESHOLD:
            ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1, fill=False, hatch="////",
                                        edgecolor="white", linewidth=0, alpha=0.35, zorder=3))

# Gold borders for SWITCH(v2a) selection cells: MSR(ledoit_wolf) in R0, MSR(sample) in R5
for col_key, regime_key in [("MSR(ledoit_wolf)", 0), ("MSR(sample)", 5)]:
    if col_key in HEATMAP_STRATS:
        ri, ci = HEATMAP_STRATS.index(col_key), reg_order.index(regime_key)
        ax.add_patch(FancyBboxPatch((ci-0.48, ri-0.48), 0.96, 0.96, boxstyle="square,pad=0",
                                     fill=False, edgecolor="gold", linewidth=2.5, zorder=5))

# Display short names on y-axis
display_names = [s.replace("ledoit_wolf", "LW").replace("(sample)", "(s)").replace("(oas)", "(OAS)") for s in HEATMAP_STRATS]
ax.set_xticks(range(8)); ax.set_xticklabels([REGIME_LABELS[k] for k in reg_order], fontsize=7.5)
ax.set_yticks(range(len(HEATMAP_STRATS))); ax.set_yticklabels(display_names, fontsize=8)
ax.set_title(
    "Regime-Conditional Annualized Sharpe — 12 Base Strategies x 8 Regimes\n"
    "(*sparse: n < 252 days; gold border = SWITCH(v2a) selection rule)",
    fontsize=9.5, fontweight="bold")
fig.tight_layout()
plt.show()


### Figure 4 — VMP Exposure Mechanism for MSR(LW)

*Paper reference: §2.4 VMP Overlay, Figure 4 caption.*
21-day realized vol (top) and exposure multiplier (bottom) for MSR(LW).


In [ ]:
msr_lw = base["MSR(ledoit_wolf)"]
target_vol_msr = msr_lw.std() * np.sqrt(TRADING_DAYS)
roll_vol_msr   = msr_lw.rolling(21).std() * np.sqrt(TRADING_DAYS)
roll_vol_lag   = roll_vol_msr.shift(1)
exposure_msr   = (target_vol_msr / roll_vol_lag).clip(0.25, 1.5)
cap_low  = exposure_msr <= 0.251
cap_high = exposure_msr >= 1.499

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, gridspec_kw={"hspace": 0.08})
ax_vol, ax_exp = axes

ax_vol.plot(roll_vol_msr.index, roll_vol_msr.values * 100,
            color=FAMILY_COLORS["Classical MV"], lw=0.9, alpha=0.85, label="21-day realized vol")
ax_vol.axhline(target_vol_msr * 100, color="black", ls="--", lw=1.0,
               label=f"Long-run vol ({target_vol_msr*100:.1f}%)")
shade_crises(ax_vol)
ax_vol.set_ylabel("Realized Vol (%, ann.)")
ax_vol.set_title("MSR(LW): 21-day Realized Volatility and VMP Exposure Multiplier",
                  fontsize=11, fontweight="bold")
ax_vol.legend(fontsize=8, loc="upper right")

ax_exp.plot(exposure_msr.index, exposure_msr.values,
            color=FAMILY_COLORS["Classical MV"], lw=0.9, alpha=0.85, label="VMP exposure")
ax_exp.fill_between(exposure_msr.index, 0.25, exposure_msr.values,
                     where=cap_low, color="#d62728", alpha=0.35, label="Vol cap active (0.25×)")
ax_exp.fill_between(exposure_msr.index, exposure_msr.values, 1.5,
                     where=cap_high, color="#2ca02c", alpha=0.35, label="Max exposure (1.5×)")
ax_exp.axhline(0.25, color="#d62728", ls="--", lw=0.9, alpha=0.7)
ax_exp.axhline(1.50, color="#2ca02c", ls="--", lw=0.9, alpha=0.7)
ax_exp.axhline(1.00, color="black",   ls=":", lw=0.7, alpha=0.5)
shade_crises(ax_exp)
fig.canvas.draw()
ylim_e = ax_exp.get_ylim()
for start, end, label in CRISES:
    ax_exp.text(start+(end-start)/2, ylim_e[0]+0.01*(ylim_e[1]-ylim_e[0]),
                label, fontsize=6.5, ha="center", va="bottom", color="dimgray", style="italic")
ax_exp.set_ylabel("Exposure multiplier"); ax_exp.set_xlabel("Date")
ax_exp.legend(fontsize=8, loc="upper right", ncol=2)
ax_exp.set_ylim(0.15, 1.65)
fig.tight_layout()
plt.show()


### Reproducibility Check

Assert every number in the paper's main table matches the values computed from cache.
Tolerance: ±0.001 for Sharpe, ±0.001 for Ann Ret (%pt), ±0.001 for Max DD (%pt).


In [ ]:
# Load full 31-strategy caches (constrained MV + long-short included)
base31_all = pd.read_parquet("data/cache/portfolio_returns/31strategies_29assets_2003_2026.parquet")
vmp31_all  = pd.read_parquet("data/cache/portfolio_returns/31strategies_vmp_29assets_2003_2026.parquet")

# Load weights for constrained/LS strategies not in original weights_cache
for _col in base31_all.columns:
    if _col not in weights_cache:
        _p = _weights_path(_col, suffix=WEIGHTS_SUFFIX)
        if _p.exists():
            weights_cache[_col] = pd.read_parquet(_p)

# 29-asset 2003-2026 ground-truth table for reproducibility checks
# (col, is_vmp): (ann_ret%, ann_vol%, sharpe, max_dd%, calmar, turnover%, net_sharpe_10bps)
PAPER_TABLE = {
    ("EW", False):               (12.60, 13.89, 0.924, -37.86, 0.333, 0.00, 0.924),
    ("EW", True):                (15.31, 13.37, 1.133, -27.32, 0.560, 0.00, 1.133),
    ("GMV(sample)", False):      ( 3.02,  3.16, 0.958,  -9.62, 0.314, 0.00, 0.944),
    ("GMV(sample)", True):       ( 3.13,  2.31, 1.345,  -8.53, 0.367, 0.00, 1.327),
    ("GMV(ledoit_wolf)", False):  ( 3.81,  4.01, 0.954, -11.09, 0.344, 0.01, 0.921),
    ("GMV(ledoit_wolf)", True):   ( 4.47,  3.65, 1.215, -12.66, 0.353, 0.01, 1.178),
    ("GMV(oas)", False):          ( 3.36,  3.64, 0.925, -10.10, 0.332, 0.00, 0.893),
    ("GMV(oas)", True):           ( 3.86,  3.18, 1.207, -11.60, 0.333, 0.00, 1.170),
    ("MSR(sample)", False):       ( 6.71,  7.58, 0.895, -19.99, 0.336, 0.05, 0.736),
    ("MSR(sample)", True):        ( 7.59,  5.78, 1.295, -10.10, 0.751, 0.05, 1.086),
    ("MSR(ledoit_wolf)", False):  (11.21, 10.56, 1.059, -19.85, 0.565, 0.05, 0.948),
    ("MSR(ledoit_wolf)", True):   (13.07, 10.34, 1.239, -21.37, 0.612, 0.05, 1.126),
    ("MDP(sample)", False):       ( 5.67,  5.13, 1.101, -17.75, 0.320, 0.03, 0.975),
    ("MDP(sample)", True):        ( 6.52,  4.70, 1.368, -12.71, 0.513, 0.03, 1.231),
    ("MDP(ledoit_wolf)", False):  ( 6.55,  5.57, 1.166, -14.80, 0.443, 0.01, 1.132),
    ("MDP(ledoit_wolf)", True):   ( 7.70,  5.52, 1.372, -13.03, 0.591, 0.01, 1.337),
    ("RP(sample)", False):        (11.65, 12.74, 0.929, -33.10, 0.352, 0.00, 0.928),
    ("RP(sample)", True):         (14.09, 12.60, 1.110, -26.23, 0.537, 0.00, 1.108),
    ("RP(ledoit_wolf)", False):   (11.62, 12.77, 0.926, -32.90, 0.353, 0.00, 0.923),
    ("RP(ledoit_wolf)", True):    (14.10, 12.62, 1.108, -26.15, 0.539, 0.00, 1.106),
    ("HRP(sample)", False):       ( 6.80,  6.50, 1.045, -16.57, 0.410, 0.04, 0.896),
    ("HRP(sample)", True):        ( 7.54,  6.36, 1.176, -15.10, 0.500, 0.04, 1.022),
    ("HRP(ledoit_wolf)", False):  ( 7.14,  6.51, 1.093, -15.65, 0.457, 0.03, 0.966),
    ("HRP(ledoit_wolf)", True):   ( 7.99,  6.40, 1.232, -13.41, 0.596, 0.03, 1.103),
    ("SWITCH(sample)", False):    ( 8.31,  8.07, 1.029, -19.04, 0.436, 0.03, 0.928),
    ("SWITCH(sample)", True):     ( 9.27,  7.05, 1.293, -13.11, 0.707, 0.03, 1.178),
    ("SWITCH(ledoit_wolf)", False): ( 9.55,  8.81, 1.080, -20.16, 0.474, 0.02, 1.023),
    ("SWITCH(ledoit_wolf)", True):  (10.60,  8.24, 1.265, -17.39, 0.610, 0.02, 1.203),
    ("TSMOM(12m)", False):         ( 5.45,  6.93, 0.801, -21.68, 0.252, 0.03, 0.704),
    ("TSMOM(12m)", True):          ( 6.98,  6.58, 1.058, -13.80, 0.506, 0.03, 0.957),
    ("TSMOM(6m)", False):          ( 7.02,  7.25, 0.971, -24.18, 0.290, 0.05, 0.812),
    ("TSMOM(6m)", True):           ( 7.70,  6.75, 1.133, -12.47, 0.618, 0.05, 0.961),
    ("BL-Eq(sample)", False):      (12.48, 13.92, 0.915, -37.86, 0.330, 0.00, 0.915),
    ("BL-Eq(sample)", True):       (15.12, 13.41, 1.118, -27.39, 0.552, 0.00, 1.118),
    ("BL-Eq(LW)", False):          (12.48, 13.92, 0.915, -37.86, 0.330, 0.00, 0.915),
    ("BL-Eq(LW)", True):           (15.12, 13.41, 1.118, -27.39, 0.552, 0.00, 1.118),
    ("BL-Mom(LW)", False):         (12.57, 12.07, 1.042, -21.34, 0.589, 0.05, 0.938),
    ("BL-Mom(LW)", True):          (14.65, 11.81, 1.217, -21.84, 0.671, 0.05, 1.111),
    ("BL-Rev(LW)", False):         (12.09, 20.34, 0.663, -51.42, 0.235, 0.09, 0.550),
    ("BL-Rev(LW)", True):          (13.95, 18.02, 0.816, -45.86, 0.304, 0.09, 0.687),
    ("FF3-Mom", False):            (11.03, 17.52, 0.685, -41.03, 0.269, 0.20, 0.405),
    ("FF3-Mom", True):             (12.37, 16.13, 0.804, -29.61, 0.418, 0.20, 0.500),
    ("FF3-LowVol", False):         ( 4.34,  4.25, 1.021, -10.68, 0.406, 0.00, 0.999),
    ("FF3-LowVol", True):          ( 4.59,  3.92, 1.165, -11.15, 0.412, 0.00, 1.141),
    ("FF3-Quality", False):        ( 7.59,  9.59, 0.811, -23.15, 0.328, 0.04, 0.715),
    ("FF3-Quality", True):         ( 8.59,  8.37, 1.027, -17.42, 0.493, 0.04, 0.916),
    ("FF3-Multi", False):          ( 7.95,  8.87, 0.907, -19.85, 0.400, 0.08, 0.691),
    ("FF3-Multi", True):           ( 8.80,  8.48, 1.037, -15.29, 0.575, 0.08, 0.812),
    ("MSR_C(ledoit_wolf)", False):  (10.52, 10.29, 1.024, -19.48, 0.540, 0.05, 0.897),
    ("MSR_C(ledoit_wolf)", True):   (12.27, 10.19, 1.187, -18.38, 0.668, 0.05, 1.058),
    ("MSR_C(sample)", False):       ( 6.82,  8.01, 0.864, -18.68, 0.365, 0.05, 0.707),
    ("MSR_C(sample)", True):        ( 7.70,  7.28, 1.055, -13.19, 0.584, 0.05, 0.882),
    ("MVO_C(ledoit_wolf)", False):  ( 2.98,  4.02, 0.750, -13.51, 0.221, 0.00, 0.721),
    ("MVO_C(ledoit_wolf)", True):   ( 3.22,  3.69, 0.877, -14.25, 0.226, 0.00, 0.845),
    ("MVO_C(sample)", False):       ( 2.95,  3.94, 0.757, -14.05, 0.210, 0.00, 0.725),
    ("MVO_C(sample)", True):        ( 2.99,  3.54, 0.850, -14.70, 0.203, 0.00, 0.814),
    ("TSMOM-LS(12m)", False):  ( 3.67,  5.86, 0.645, -16.42, 0.224, 0.02, 0.550),
    ("TSMOM-LS(12m)", True):   ( 4.48,  5.44, 0.833, -14.29, 0.313, 0.02, 0.730),
    ("BL-Mom-LS(LW)", False):  ( 4.18,  4.65, 0.904, -11.87, 0.352, 0.04, 0.665),
    ("BL-Mom-LS(LW)", True):   ( 4.78,  4.58, 1.042, -11.95, 0.400, 0.04, 0.799),
    ("FF3-Mom-LS", False):     ( 0.53,  8.99, 0.103, -26.97, 0.020, 0.14, -0.290),
    ("FF3-Mom-LS", True):      (-0.73,  8.99, -0.037, -35.32, -0.021, 0.14, -0.430),
}

TOL_SHARPE = 0.003
TOL_PCT    = 0.05   # percentage points

failures = []
for (col, is_vmp), (exp_ret, exp_vol, exp_sharpe, exp_maxdd, exp_calmar, exp_to, exp_net) in PAPER_TABLE.items():
    s = vmp31_all[f"VMP({col})"] if is_vmp else base31_all[col]
    act_sharpe = ann_sharpe(s)
    act_ret    = ann_return(s) * 100
    act_maxdd  = max_dd(s) * 100
    w = weights_cache.get(col)
    act_net = ann_sharpe(apply_costs(s, w)) if w is not None else act_sharpe
    errs = []
    if abs(act_sharpe - exp_sharpe) > TOL_SHARPE:
        errs.append(f"Sharpe: paper={exp_sharpe:.3f} got={act_sharpe:.3f}")
    if abs(act_ret - exp_ret) > TOL_PCT:
        errs.append(f"AnnRet: paper={exp_ret:.2f}% got={act_ret:.2f}%")
    if abs(act_maxdd - exp_maxdd) > TOL_PCT:
        errs.append(f"MaxDD: paper={exp_maxdd:.2f}% got={act_maxdd:.2f}%")
    if abs(act_net - exp_net) > TOL_SHARPE:
        errs.append(f"NetSharpe: paper={exp_net:.3f} got={act_net:.3f}")
    if errs:
        failures.append(f"{'VMP(' + col + ')' if is_vmp else col}: {chr(59).join(errs)}")

if failures:
    print("REPRODUCIBILITY FAILURES:")
    for f in failures:
        print(f"  x {f}")
    raise AssertionError(f"{len(failures)} reconciliation failures")
else:
    n_base = sum(1 for (_, iv) in PAPER_TABLE if not iv)
    n_vmp  = sum(1 for (_, iv) in PAPER_TABLE if iv)
    print(f"checkmark {len(PAPER_TABLE)} records ({n_base} base + {n_vmp} VMP) reconciled "
          f"within tolerance (Sharpe +/-{TOL_SHARPE}, AnnRet/MaxDD +/-{TOL_PCT}%pt)")


---
## Part 2 — Practitioner-Standard Analytics

Extensions not in the paper.

**Arc:** performance (§2.1–2.2) → risk (§2.3–2.5) → implementation (§2.6–2.7) → allocations (§2.8–2.10) → cross-strategy structure (§2.11) → statistical robustness (§2.12–2.13). Each section adds a lens that the headline Sharpe ratio can't supply — from tail behavior to concentration to whether the findings survive formal scrutiny.

### 2.1 — Rolling 12-Month Sharpe Time Series

Six representative strategies: EW, GMV(LW), MSR(LW), MDP(LW), SWITCH(v2a), VMP(BL-Mom(LW)).
Rolling window: 252 trading days.


In [ ]:
REP6_COLS = [
    ("EW",                   "EW",             FAMILY_COLORS["EW (benchmark)"]),
    ("GMV(ledoit_wolf)",      "GMV(LW)",        FAMILY_COLORS["Classical MV"]),
    ("MSR(ledoit_wolf)",      "MSR(LW)",        FAMILY_COLORS["Classical MV"]),
    ("MDP(ledoit_wolf)",      "MDP(LW)",        FAMILY_COLORS["Diversification"]),
    (None,                    "SWITCH(v2a)",    FAMILY_COLORS["Regime"]),
    ("VMP(BL-Mom(LW))",       "VMP(BL-Mom(LW))",FAMILY_COLORS["Black-Litterman"]),
]

WINDOW = TRADING_DAYS  # 252 days

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey=True)
axes = axes.flatten()

for idx, (col, label, color) in enumerate(REP6_COLS):
    ax = axes[idx]
    s = switch_v2a if col is None else strategy_returns(col)
    roll_sharpe = s.rolling(WINDOW).apply(lambda x: x.mean() / x.std() * np.sqrt(TRADING_DAYS), raw=True)
    ax.plot(roll_sharpe.index, roll_sharpe.values, color=color, lw=1.0, alpha=0.9)
    ax.axhline(0, color="black", lw=0.6, ls=":")
    ax.axhline(1, color="black", lw=0.6, ls="--", alpha=0.5)
    shade_crises(ax, alpha=0.08)
    ax.set_title(label, fontsize=9, fontweight="bold", color=color)
    ax.set_ylabel("Rolling 12m Sharpe")
    ax.set_xlabel("Date")
    full_mean = ann_sharpe(s)
    ax.text(0.02, 0.97, f"Full-period: {full_mean:.2f}", transform=ax.transAxes,
            fontsize=7.5, va="top", color=color)

fig.suptitle("Rolling 12-Month Sharpe — 6 Representative Strategies", fontsize=11, fontweight="bold")
fig.tight_layout()
plt.show()


> **Reading — §2.1:** Rolling Sharpe reveals whether performance is durable or regime-clustered — a strong full-sample number earned mostly in one bull run looks nothing like a consistently positive rolling series. VMP raises the floor by scaling down during the high-vol episodes that produce the deepest negative windows. 12-month windows are too noisy for precision comparisons; use this for qualitative regime diagnosis. *See Appendix B (5-year buckets); Finding 6.*

### 2.2 — Calendar-Year Returns Heatmap

24 base strategies × 19 calendar years (2008–2026, with 2026 partial through end of data).
Diverging colormap centered at 0%.


In [ ]:
years = sorted(base.index.year.unique())
cal_data = {}
for col in base.columns:
    s = base[col]
    yearly = {}
    for yr in years:
        mask = s.index.year == yr
        sub  = s[mask].dropna()
        if len(sub) > 0:
            yearly[yr] = ((1 + sub).prod() - 1) * 100
        else:
            yearly[yr] = np.nan
    cal_data[DISPLAY[col]] = yearly

df_cal = pd.DataFrame(cal_data).T   # strategies × years
df_cal.columns = [str(y) for y in df_cal.columns]

# Sort strategies by median annual return
strategy_order = df_cal.median(axis=1).sort_values(ascending=False).index

vmax_cal = max(abs(np.nanpercentile(df_cal.values, 2.5)),
               abs(np.nanpercentile(df_cal.values, 97.5)))
vmax_cal = min(vmax_cal, 60.0)

fig, ax = plt.subplots(figsize=(18, 9))
im = ax.imshow(df_cal.loc[strategy_order].values, cmap="RdBu", vmin=-vmax_cal, vmax=vmax_cal,
               aspect="auto")
plt.colorbar(im, ax=ax, label="Annual Return (%)", shrink=0.6)

for i, strat in enumerate(strategy_order):
    for j, yr in enumerate(df_cal.columns):
        val = df_cal.loc[strat, yr]
        if not np.isnan(val):
            color = "white" if abs(val) > vmax_cal * 0.5 else "black"
            ax.text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=6.5, color=color)

ax.set_xticks(range(len(df_cal.columns)))
ax.set_xticklabels(df_cal.columns, fontsize=7.5, rotation=45, ha="right")
ax.set_yticks(range(len(strategy_order)))
ax.set_yticklabels(strategy_order, fontsize=7.5)
ax.set_title("Calendar-Year Returns (%) — 24 Base Strategies × 19 Years",
              fontsize=11, fontweight="bold")
fig.tight_layout()
plt.show()


> **Reading — §2.2:** Calendar years expose consistency — the quality that separates "reliably good" from "average on paper, volatile in practice." Strategies that avoided 2008 and 2022 deep-red squares did so through structural defensiveness (GMV, FF3-LowVol), not timing. Calendar boundaries are arbitrary (a December crash looks better than a February one); pair this with Appendix B's 5-year sub-periods. *See §2.3 for the finer-grained loss picture.*

### 2.3 — Underwater Drawdown Plot

Top 6 strategies by Sharpe (from all 48): cumulative wealth (top) and underwater drawdown (bottom).
Paired 2-panel layout, one column per strategy.


In [ ]:
top6_cols = df_all.nlargest(6, "sharpe")["col"].tolist()
top6_labels = [c.replace("ledoit_wolf","LW").replace("(oas)","(OAS)") for c in top6_cols]
top6_rets = [vmp[c] if c in vmp.columns else base[c] for c in top6_cols]
top6_colors = [FAMILY_COLORS[FAMILY_MAP.get(c.replace("VMP(","").rstrip(")"), "EW (benchmark)")]
               for c in top6_cols]

fig, axes = plt.subplots(2, 6, figsize=(18, 7), sharex=True)

for idx in range(6):
    s = top6_rets[idx]
    cw = cum_wealth(s)
    dd = (cw - cw.cummax()) / cw.cummax() * 100
    color = top6_colors[idx]
    label = top6_labels[idx]
    sharpe_val = ann_sharpe(s)

    ax_top = axes[0, idx]
    ax_bot = axes[1, idx]

    ax_top.semilogy(cw.index, cw.values, color=color, lw=1.1)
    shade_crises(ax_top, alpha=0.07)
    ax_top.set_title(f"{label}\nSharpe={sharpe_val:.3f}", fontsize=7.5, fontweight="bold", color=color)
    if idx == 0:
        ax_top.set_ylabel("Cumulative Wealth (log)")

    ax_bot.fill_between(dd.index, dd.values, 0, where=(dd.values < 0),
                         color=color, alpha=0.4)
    ax_bot.plot(dd.index, dd.values, color=color, lw=0.7)
    ax_bot.axhline(0, color="black", lw=0.5)
    shade_crises(ax_bot, alpha=0.07)
    if idx == 0:
        ax_bot.set_ylabel("Drawdown (%)")
    ax_bot.set_xlabel("")

fig.suptitle("Top-6 Strategies by Sharpe — Cumulative Wealth & Underwater Drawdown",
              fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


> **Reading — §2.3:** Recovery time matters as much as drawdown depth — a 20% loss with a 4-year recovery is harder to hold than a 30% loss that heals in 18 months; the former forces capitulation at the trough. VMP compresses both depth and duration. Long-only constraint means all equity-heavy strategies spend extended time underwater in 2008–2009 regardless of construction; genuine differentiation comes from variance-minimizing and defensive strategies. *See Finding 9 (BL-Mom 50.85% drawdown); §3.2 Sharpe vs. drawdown scatter.*

### 2.4 — CVaR(95%) Bar Chart

All 24 base strategies, sorted descending by CVaR (tail risk). Strategies with high CVaR relative
to their Sharpe ratio are flagged.


In [ ]:
def cvar_95(s):
    """Historical CVaR at 95% confidence (expected loss in worst 5% of days)."""
    cutoff = np.percentile(s, 5)
    tail   = s[s <= cutoff]
    return tail.mean() * 100  # in %

cvar_rows = []
for col in base.columns:
    s = base[col]
    cv = cvar_95(s)
    sh = ann_sharpe(s)
    cvar_rows.append({"col": col, "display": DISPLAY[col],
                       "cvar_95": cv, "sharpe": sh,
                       "family": FAMILY_MAP.get(col, "EW (benchmark)")})
df_cvar = pd.DataFrame(cvar_rows).sort_values("cvar_95")  # worst CVaR first (most negative)

# Flag if |CVaR| / Sharpe > threshold (high loss per unit of Sharpe)
df_cvar["flag"] = df_cvar["cvar_95"].abs() / df_cvar["sharpe"] > 2.5

fig, ax = plt.subplots(figsize=(13, 7))
colors_bar = [FAMILY_COLORS[f] for f in df_cvar["family"]]
bars = ax.barh(df_cvar["display"], df_cvar["cvar_95"], color=colors_bar, edgecolor="white", lw=0.3)

for bar, row in zip(bars, df_cvar.itertuples()):
    if row.flag:
        ax.text(bar.get_width() - 0.05, bar.get_y() + bar.get_height()/2,
                "★", va="center", ha="right", fontsize=9, color="gold")

ax.axvline(0, color="black", lw=0.7)
ax.set_xlabel("CVaR(95%) — Daily Return (%)")
ax.set_title("CVaR(95%) by Base Strategy — Sorted by Tail Risk\n★ = High loss-per-Sharpe (|CVaR|/Sharpe > 2.5)",
              fontsize=10, fontweight="bold")

legend_patches = [mpatches.Patch(color=FAMILY_COLORS[f], label=f) for f in FAMILY_COLORS]
ax.legend(handles=legend_patches, ncol=2, fontsize=7, loc="lower right")
fig.tight_layout()
plt.show()


> **Reading — §2.4:** CVaR(95%) reveals tail risk that Sharpe and even max drawdown can hide. Strategies with high CVaR/Sharpe ratios (BL-Mom, BL-Rev) carry concentrated downside not captured by their headline numbers — consistent with Finding 9's flag that BL-Mom's path is unsuitable for most risk budgets without drawdown stops. Low-vol strategies (GMV(sample), FF3-LowVol) have low CVaR by construction but their low absolute return makes the favorable CVaR/Sharpe profile less actionable without leverage. The flagged strategies are not necessarily bad — they're strategies where one-number summaries undersell the tail.

### 2.5 — Rolling 252-Day Correlation to SPY

6 representative strategies vs SPY.US returns (from prices_29.parquet).
Higher correlation = more equity beta exposure.


In [ ]:
spy_aligned = spy_ret.reindex(base.index).ffill()

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey=True, sharex=True)
axes = axes.flatten()

for idx, (col, label, color) in enumerate(REP6_COLS):
    ax = axes[idx]
    s = switch_v2a if col is None else strategy_returns(col)
    common_idx = s.index.intersection(spy_aligned.dropna().index)
    s_c   = s.reindex(common_idx)
    spy_c = spy_aligned.reindex(common_idx)
    # Vectorized rolling correlation via pandas
    combined = pd.DataFrame({"strat": s_c, "spy": spy_c})
    roll_corr = combined["strat"].rolling(WINDOW).corr(combined["spy"])
    ax.plot(roll_corr.index, roll_corr.values, color=color, lw=1.0, alpha=0.9)
    ax.axhline(0, color="black", lw=0.6, ls=":")
    ax.axhline(0.5, color="black", lw=0.5, ls="--", alpha=0.4)
    shade_crises(ax, alpha=0.08)
    ax.set_title(label, fontsize=9, fontweight="bold", color=color)
    ax.set_ylabel("Rolling Corr to SPY")
    ax.set_ylim(-0.6, 1.05)
    full_corr = s_c.corr(spy_c)
    ax.text(0.02, 0.03, f"Full-period: {full_corr:.2f}", transform=ax.transAxes,
            fontsize=7.5, color=color)

fig.suptitle("Rolling 252-Day Correlation to SPY — 6 Representative Strategies",
              fontsize=11, fontweight="bold")
fig.tight_layout()
plt.show()


> **Reading — §2.5:** The relevant measure is *crisis* correlation, not unconditional correlation. A strategy can show low average beta to SPY while spiking toward 1.0 in every stress episode — exactly when diversification matters most. SPY is a single-factor proxy; low SPY correlation does not rule out credit, duration, or liquidity risk loading. *See Finding 11 (FF3-LowVol low-beta profile); §2.8 (asset-class pies).*

### 2.6 — Turnover Histogram + Boxplot

All 24 base strategies. Histogram of daily one-way turnover distribution; boxplot sorted by median.


In [ ]:
turnover_series = {}
for col in base.columns:
    w = weights_cache.get(col)
    if w is not None:
        ts = compute_turnover(w).dropna() * 100  # in %
        turnover_series[DISPLAY[col]] = ts

# Sort by median
median_to = {k: v.median() for k, v in turnover_series.items()}
sorted_by_median = sorted(median_to, key=median_to.get)

fig, (ax_hist, ax_box) = plt.subplots(1, 2, figsize=(16, 8))

# Histogram — all strategies overlaid
for col, ts in turnover_series.items():
    fam = FAMILY_MAP.get([k for k, v in DISPLAY.items() if v == col][0], "EW (benchmark)") if col != "SWITCH(v2a)" else "Regime"
    ax_hist.hist(ts.values, bins=50, alpha=0.25, color=FAMILY_COLORS.get(fam, "gray"), density=True)
ax_hist.set_xlabel("Daily One-Way Turnover (%)")
ax_hist.set_ylabel("Density")
ax_hist.set_title("Turnover Distribution — 24 Base Strategies (overlaid)", fontsize=9, fontweight="bold")
ax_hist.set_xlim(left=0)

# Boxplot — sorted by median
box_data  = [turnover_series[k].values for k in sorted_by_median]
box_cols  = []
for k in sorted_by_median:
    orig_col = [c for c, d in DISPLAY.items() if d == k]
    box_cols.append(FAMILY_COLORS.get(FAMILY_MAP.get(orig_col[0], "EW (benchmark)"), "gray") if orig_col else "gray")

bplot = ax_box.boxplot(box_data, vert=False, patch_artist=True,
                        medianprops=dict(color="black", lw=1.5),
                        flierprops=dict(marker=".", markersize=2, alpha=0.3),
                        whis=[5, 95])
for patch, c in zip(bplot["boxes"], box_cols):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)

ax_box.set_yticks(range(1, len(sorted_by_median)+1))
ax_box.set_yticklabels(sorted_by_median, fontsize=7.5)
ax_box.set_xlabel("Daily One-Way Turnover (%)")
ax_box.set_title("Turnover Boxplot — Sorted by Median (5th–95th pctile whiskers)",
                  fontsize=9, fontweight="bold")
ax_box.set_xlim(left=0)

fig.tight_layout()
plt.show()


> **Reading — §2.6:** Turnover bridges gross to net Sharpe. What matters is the distribution's tail: a strategy at median 5% daily turnover with occasional 40% rotation spikes has a worse cost profile than one at 5% uniformly. Monthly rebalancing captures only month-end weight changes; VMP's daily scaling adds intra-month turnover not measured here. *See §3.3 Transaction-Cost Sensitivity; Finding 13 (high-turnover collapsers).*

### 2.7 — Maximum Single-Position Weight Distribution

Boxplot of daily max-weight across the 30-asset portfolio, all 24 base strategies.
Sorted by median max weight.


In [ ]:
max_weight_series = {}
for col in base.columns:
    w = weights_cache.get(col)
    if w is not None:
        max_weight_series[DISPLAY[col]] = w.max(axis=1).dropna() * 100  # in %

median_mw = {k: v.median() for k, v in max_weight_series.items()}
sorted_mw = sorted(median_mw, key=median_mw.get)

box_data_mw = [max_weight_series[k].values for k in sorted_mw]
box_cols_mw = []
for k in sorted_mw:
    orig = [c for c, d in DISPLAY.items() if d == k]
    box_cols_mw.append(FAMILY_COLORS.get(FAMILY_MAP.get(orig[0], "EW (benchmark)"), "gray") if orig else "gray")

fig, ax = plt.subplots(figsize=(12, 9))
bplot = ax.boxplot(box_data_mw, vert=False, patch_artist=True,
                   medianprops=dict(color="black", lw=1.5),
                   flierprops=dict(marker=".", markersize=2, alpha=0.3),
                   whis=[5, 95])
for patch, c in zip(bplot["boxes"], box_cols_mw):
    patch.set_facecolor(c); patch.set_alpha(0.7)

ax.set_yticks(range(1, len(sorted_mw)+1))
ax.set_yticklabels(sorted_mw, fontsize=7.5)
ax.axvline(1/30*100, color="black", ls="--", lw=0.8, alpha=0.5)
ax.text(1/30*100 + 0.2, len(sorted_mw)*0.9, "EW (3.33%)", fontsize=7, color="dimgray")
ax.set_xlabel("Max Single-Position Weight (%)")
ax.set_title("Maximum Single-Position Weight Distribution — 24 Base Strategies\n(5th–95th pctile whiskers; dashed = EW benchmark 3.33%)",
              fontsize=9, fontweight="bold")

legend_patches = [mpatches.Patch(color=FAMILY_COLORS[f], label=f) for f in FAMILY_COLORS]
ax.legend(handles=legend_patches, ncol=2, fontsize=7.5, loc="lower right")
fig.tight_layout()
plt.show()


> **Reading — §2.7:** Max single-position weight is the most direct concentration-risk diagnostic — Sharpe and vol statistics obscure it. A strategy that periodically collapses to a single asset is a single-stock bet with extra steps for that period. GMV(sample) and MSR(sample) are the canonical cases (Findings 1–2); their favorable headline Sharpe numbers are concentration artifacts. *See §2.8 (average allocation pies).*

### 2.8 — Average Weight Pie Charts

Time-averaged allocation for 6 representative strategies: EW, GMV(sample), GMV(LW), MSR(LW), MDP(LW), VMP(SWITCH(LW)).
Assets colored by asset class. 2×3 grid.


In [ ]:
ASSET_CLASS = {
    "AAPL.US": "US Equity", "MSFT.US": "US Equity", "GOOGL.US": "US Equity",
    "NVDA.US": "US Equity", "JPM.US": "US Equity", "JNJ.US": "US Equity",
    "XOM.US": "US Equity", "WMT.US": "US Equity",
    "XLK.US": "Sector ETF", "XLF.US": "Sector ETF", "XLE.US": "Sector ETF",
    "XLV.US": "Sector ETF", "XLP.US": "Sector ETF", "XLU.US": "Sector ETF",
    "SPY.US": "Broad Equity", "IWM.US": "Broad Equity",
    "EFA.US": "Intl Equity", "EEM.US": "Intl Equity", "FXI.US": "Intl Equity",
    "SHY.US": "Fixed Income", "IEF.US": "Fixed Income", "TLT.US": "Fixed Income",
    "AGG.US": "Fixed Income", "HYG.US": "Fixed Income",
    "GLD.US": "Alternatives", "SLV.US": "Alternatives", "DBC.US": "Alternatives",
    "USO.US": "Alternatives", "EURUSD.FOREX": "Alternatives", "BTC-USD.CC": "Alternatives",
}
AC_COLORS = {
    "US Equity":    "#1f77b4",
    "Sector ETF":   "#aec7e8",
    "Broad Equity": "#2ca02c",
    "Intl Equity":  "#98df8a",
    "Fixed Income": "#ff7f0e",
    "Alternatives": "#d62728",
}

PIE6 = [
    ("EW",               "EW"),
    ("GMV(sample)",      "GMV(sample)"),
    ("GMV(ledoit_wolf)", "GMV(LW)"),
    ("MSR(ledoit_wolf)", "MSR(LW)"),
    ("MDP(ledoit_wolf)", "MDP(LW)"),
    ("SWITCH(ledoit_wolf)", "VMP(SWITCH(LW))"),  # use SWITCH(LW) weights
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, (wt_col, label) in enumerate(PIE6):
    ax = axes[idx]
    w = weights_cache.get(wt_col)
    if w is None:
        ax.text(0.5, 0.5, "no data", ha="center", va="center")
        continue
    avg_w = w.mean(axis=0).dropna()
    avg_w = avg_w[avg_w > 0.001]  # filter tiny weights

    # Group by asset class
    ac_weights = {}
    asset_labels = []
    asset_colors_pie = []
    for ticker, wgt in avg_w.items():
        ac = ASSET_CLASS.get(ticker, "Alternatives")
        ac_weights[ac] = ac_weights.get(ac, 0) + wgt
    acs = sorted(ac_weights, key=ac_weights.get, reverse=True)
    wgts = [ac_weights[ac] for ac in acs]
    colors_pie = [AC_COLORS.get(ac, "gray") for ac in acs]
    wedge_labels = [f"{ac}\n{ac_weights[ac]*100:.1f}%" for ac in acs]
    ax.pie(wgts, labels=wedge_labels, colors=colors_pie, startangle=90,
           textprops={"fontsize": 6.5}, pctdistance=0.8,
           wedgeprops=dict(edgecolor="white", lw=0.5))
    ax.set_title(label, fontsize=9, fontweight="bold",
                 color=FAMILY_COLORS.get(FAMILY_MAP.get(wt_col, "EW (benchmark)"), "black"))

fig.suptitle("Average Asset-Class Allocation — 6 Representative Strategies",
              fontsize=11, fontweight="bold")
fig.tight_layout()
plt.show()


> **Reading — §2.8:** Average allocation pies expose the structural market bet each strategy embeds. GMV variants and FF3-LowVol are structurally overweight fixed income — attractive in bond bull markets, painful in 2022. Time-averaging hides regime-conditional tilts; the crisis snapshots in §2.10 reveal the more informative picture. *See Finding 1 (GMV SHY concentration); §2.9 (allocation timeline).*

### 2.9 — Asset-Class Allocation Timeline

6 asset classes stacked area chart for the same 6 representative strategies.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=True)
axes = axes.flatten()
AC_ORDER = ["US Equity","Sector ETF","Broad Equity","Intl Equity","Fixed Income","Alternatives"]
AC_COLORS_TIMELINE = {
    "US Equity":    "#1f77b4",
    "Sector ETF":   "#aec7e8",
    "Broad Equity": "#2ca02c",
    "Intl Equity":  "#98df8a",
    "Fixed Income": "#ff7f0e",
    "Alternatives": "#d62728",
}

for idx, (wt_col, label) in enumerate(PIE6):
    ax = axes[idx]
    w = weights_cache.get(wt_col)
    if w is None:
        continue
    # Resample to monthly for smoother timeline
    w_monthly = w.resample("ME").last().fillna(0)
    ac_ts = pd.DataFrame({ac: w_monthly[[c for c in w_monthly.columns if ASSET_CLASS.get(c) == ac]].sum(axis=1)
                           for ac in AC_ORDER})
    # Fill any NaN with 0 and renormalize
    ac_ts = ac_ts.fillna(0)
    total = ac_ts.sum(axis=1).replace(0, np.nan)
    ac_ts_norm = ac_ts.div(total, axis=0).fillna(0)

    bottom = np.zeros(len(ac_ts_norm))
    for ac in AC_ORDER:
        ax.fill_between(ac_ts_norm.index, bottom, bottom + ac_ts_norm[ac].values,
                         alpha=0.85, color=AC_COLORS_TIMELINE[ac], label=ac)
        bottom += ac_ts_norm[ac].values
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("Weight")
    fam_col = FAMILY_COLORS.get(FAMILY_MAP.get(wt_col, "EW (benchmark)"), "black")
    ax.set_title(label, fontsize=9, fontweight="bold", color=fam_col)
    shade_crises(ax, alpha=0.07)

axes[0].legend(loc="upper right", fontsize=6.5, ncol=2)
fig.suptitle("Asset-Class Allocation Timeline (Monthly) — 6 Representative Strategies",
              fontsize=11, fontweight="bold")
fig.tight_layout()
plt.show()


> **Reading — §2.9:** The timeline distinguishes continuous signal-driven drift (MDP, HRP, RP) from discrete regime jumps (SWITCH variants). Hard switches are correct when the classifier is reliable but introduce implementation risk when the regime misfires — each transition accumulates turnover visible in §2.6. The US equity tilt in the universe compresses the apparent asset-class diversity. *See Finding 5 (SWITCH v2a logic); §2.10 (crisis snapshots).*

### 2.10 — Crisis-Date Allocation Snapshots

Horizontal bar charts of portfolio weights at 3 crisis dates: 2009-03, 2020-03, 2022-10.
Same 6 representative strategies. Assets colored by asset class.


In [ ]:
CRISIS_DATES = [
    pd.Timestamp("2009-03-31"),
    pd.Timestamp("2020-03-31"),
    pd.Timestamp("2022-10-31"),
]
CRISIS_LABELS = ["Mar 2009 (GFC trough)", "Mar 2020 (COVID)", "Oct 2022 (Rate Shock peak)"]

# Use SWITCH(LW) weights for "VMP(SWITCH(LW))"
SNAPSHOT6 = [
    ("EW",               "EW"),
    ("GMV(sample)",      "GMV(sample)"),
    ("GMV(ledoit_wolf)", "GMV(LW)"),
    ("MSR(ledoit_wolf)", "MSR(LW)"),
    ("MDP(ledoit_wolf)", "MDP(LW)"),
    ("SWITCH(ledoit_wolf)", "SWITCH(LW)"),
]

fig, axes = plt.subplots(3, 6, figsize=(22, 11))

for date_idx, (crisis_date, crisis_label) in enumerate(zip(CRISIS_DATES, CRISIS_LABELS)):
    for strat_idx, (wt_col, strat_label) in enumerate(SNAPSHOT6):
        ax = axes[date_idx, strat_idx]
        w = weights_cache.get(wt_col)
        if w is None:
            ax.axis("off"); continue
        # Find nearest available date
        available = w.index[w.index <= crisis_date]
        if len(available) == 0:
            ax.axis("off"); continue
        snap_date = available[-1]
        snap_w = w.loc[snap_date].dropna()
        snap_w = snap_w[snap_w > 0.001].sort_values(ascending=True)
        ac_colors_snap = [AC_COLORS.get(ASSET_CLASS.get(t, "Alternatives"), "gray")
                           for t in snap_w.index]
        ax.barh(range(len(snap_w)), snap_w.values * 100, color=ac_colors_snap, edgecolor="white", lw=0.2)
        ax.set_yticks(range(len(snap_w)))
        ax.set_yticklabels([t.replace(".US","").replace(".FOREX","").replace("-USD.CC","") for t in snap_w.index],
                            fontsize=5.5)
        ax.set_xlabel("Weight (%)", fontsize=6)
        if date_idx == 0:
            fam_col = FAMILY_COLORS.get(FAMILY_MAP.get(wt_col, "EW (benchmark)"), "black")
            ax.set_title(strat_label, fontsize=7.5, fontweight="bold", color=fam_col)
        if strat_idx == 0:
            ax.set_ylabel(crisis_label, fontsize=7, fontweight="bold")

fig.suptitle("Portfolio Composition at Crisis Dates — 6 Strategies", fontsize=11, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


> **Reading — §2.10:** Snapshots test the counterfactual that matters: was the strategy defensive when it needed to be? Weights shown are month-end prior — no hindsight. Three stress episodes is a thin validation sample; the regime-conditional Sharpe heatmap (Figure 3) is the more principled test of defensive reliability. *See Finding 9 (BL-Mom's undefended drawdown); Figure 3.*

### 2.11 — 48×48 Strategy Correlation Matrix (Hierarchically Clustered)

Daily-return correlation matrix for all 48 strategies (24 base + 24 VMP variants).
Hierarchical clustering (Ward linkage) reorders rows/columns by cluster.


In [ ]:
# Combine all 48 strategies into one DataFrame
all_rets = pd.concat([base.rename(columns={c: DISPLAY[c] for c in base.columns}),
                       vmp.rename(columns={c: c.replace("ledoit_wolf","LW").replace("(oas)","(OAS)") for c in vmp.columns})],
                      axis=1)

# Column family for color stripe
def col_family(name):
    raw = name.replace("VMP(","").rstrip(")")
    raw_orig = {v: k for k, v in DISPLAY.items()}.get(raw, raw)
    return FAMILY_MAP.get(raw_orig, "EW (benchmark)")

corr_mat = all_rets.corr()

# Hierarchical clustering on distance = 1 - corr
dist = squareform(1 - corr_mat.values, checks=False)
dist = np.clip(dist, 0, None)
link = linkage(dist, method="ward")
order = leaves_list(link)

corr_ordered = corr_mat.values[np.ix_(order, order)]
names_ordered = corr_mat.columns[order].tolist()

fig, ax = plt.subplots(figsize=(15, 13))
im = ax.imshow(corr_ordered, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, shrink=0.7, label="Pearson Correlation")
ax.set_xticks(range(len(names_ordered)))
ax.set_yticks(range(len(names_ordered)))
ax.set_xticklabels(names_ordered, rotation=90, fontsize=4.5)
ax.set_yticklabels(names_ordered, fontsize=4.5)
ax.set_title("48×48 Strategy Return Correlation Matrix (Ward-Clustered)",
              fontsize=11, fontweight="bold")

# Family color stripe on left
fam_stripe_colors = [FAMILY_COLORS.get(col_family(n), "gray") for n in names_ordered]
for i, c in enumerate(fam_stripe_colors):
    ax.add_patch(plt.Rectangle((-1.6, i-0.5), 1, 1, color=c, clip_on=False, transform=ax.transData, zorder=5))

legend_patches = [mpatches.Patch(color=FAMILY_COLORS[f], label=f) for f in FAMILY_COLORS]
ax.legend(handles=legend_patches, fontsize=7, loc="lower right",
          bbox_to_anchor=(1.15, 0), framealpha=0.85)
fig.tight_layout()
plt.show()


> **Reading — §2.11:** High within-cluster correlation means two strategies carry the same structural exposure — choosing between them is a tiebreaker, not a diversification decision. VMP variants cluster tightly with their base strategies because VMP is a scaling transformation, not a structural diversifier. Correlation is linear and full-sample; tail dependence in stress periods is materially higher than these numbers suggest. *See Finding 12 (VMP and switching share variance explained); §2.5.*

### 2.12 — Bootstrap Sharpe Confidence Intervals

Block bootstrap (block size = 252 days, 10,000 resamples) for the top-10 strategies by Sharpe.
Forest plot with point estimates and 95% confidence intervals.


In [ ]:
np.random.seed(42)

N_BOOT   = 10_000
BLOCK_SZ = 252

def block_bootstrap_sharpe(s, n_boot=N_BOOT, block_size=BLOCK_SZ):
    arr = s.dropna().values
    n   = len(arr)
    n_blocks = int(np.ceil(n / block_size))
    sharpes  = np.empty(n_boot)
    for b in range(n_boot):
        starts = np.random.randint(0, n - block_size + 1, size=n_blocks)
        sample = np.concatenate([arr[s:s+block_size] for s in starts])[:n]
        mu, sigma = sample.mean(), sample.std()
        sharpes[b] = mu / sigma * np.sqrt(TRADING_DAYS) if sigma > 0 else np.nan
    return sharpes

top10_strats = df_all.nlargest(10, "sharpe")[["col","display","sharpe","is_vmp"]].reset_index(drop=True)

print("Computing block bootstrap Sharpe CIs for top-10 strategies (may take ~30 seconds)...")
boot_results = []
for _, row in top10_strats.iterrows():
    s = vmp[row.col] if row.is_vmp else base[row.col]
    dist_boots = block_bootstrap_sharpe(s)
    lo, hi = np.nanpercentile(dist_boots, [2.5, 97.5])
    boot_results.append({"display": row.display, "point": row.sharpe, "lo": lo, "hi": hi})
    print(f"  {row.display:35s}  {row.sharpe:.3f}  [{lo:.3f}, {hi:.3f}]")

df_boot = pd.DataFrame(boot_results).sort_values("point")

fig, ax = plt.subplots(figsize=(9, 7))
for i, row in enumerate(df_boot.itertuples()):
    is_vmp_strat = "VMP" in row.display
    color = "#1f77b4" if is_vmp_strat else "#7f7f7f"
    ax.plot([row.lo, row.hi], [i, i], color=color, lw=2.5, alpha=0.8, solid_capstyle="butt")
    ax.scatter([row.point], [i], color=color, s=60, zorder=5)
    ax.text(row.hi + 0.005, i, f"{row.point:.3f}", va="center", fontsize=8)

ax.set_yticks(range(len(df_boot)))
ax.set_yticklabels(df_boot["display"].tolist(), fontsize=8.5)
ax.axvline(1.0, color="black", ls="--", lw=0.8, alpha=0.5)
ax.text(1.002, len(df_boot)-0.5, "Sharpe=1.0", fontsize=7, color="dimgray")
ax.set_xlabel("Annualized Sharpe Ratio")
ax.set_title(f"Block Bootstrap Sharpe 95% CIs — Top-10 Strategies\n"
             f"(block size={BLOCK_SZ} days, {N_BOOT:,} resamples)",
             fontsize=10, fontweight="bold")

vmp_patch = mpatches.Patch(color="#1f77b4", label="VMP variant")
base_patch = mpatches.Patch(color="#7f7f7f", label="Base strategy")
ax.legend(handles=[vmp_patch, base_patch], fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()


> **Reading — §2.12:** Block bootstrap with 252-day blocks preserves the regime-length autocorrelation that naive t-intervals miss. Wide confidence intervals mean the headline Sharpe is consistent with very different underlying performance — the interval is the credible range, not the point estimate. Bootstrap CIs capture sampling uncertainty along one realized path; model risk and parameter uncertainty through portfolio construction are additional layers not shown. *See §4 Statistical Robustness; Finding R1–R3.*

### 2.13 — Pairwise Sharpe-Difference Significance (Memmel 2003)

Five key contrasts tested with the Memmel (2003) / Jobson-Korkie paired Sharpe test.
H₀: Sharpe(A) = Sharpe(B).  Two-sided z-test.


In [ ]:
from scipy.stats import norm as scipy_norm

def memmel_test(r1, r2):
    """Memmel (2003) test for equality of Sharpe ratios.
    Returns (z_stat, p_value, delta_sharpe).
    Uses daily (non-annualized) Sharpe to match the original formulation.
    """
    n  = min(len(r1), len(r2))
    r1 = np.asarray(r1.dropna().iloc[:n])
    r2 = np.asarray(r2.dropna().iloc[:n])
    mu1, mu2     = r1.mean(), r2.mean()
    sigma1, sigma2 = r1.std(), r2.std()
    rho = np.corrcoef(r1, r2)[0, 1]
    sr1 = mu1 / sigma1
    sr2 = mu2 / sigma2
    var = 2 * (1 - rho) + 0.5 * (sr1**2 + sr2**2) - sr1 * sr2 * rho
    var = max(var, 1e-12)
    z   = np.sqrt(n) * (sr1 - sr2) / np.sqrt(var)
    p   = 2 * scipy_norm.sf(abs(z))
    delta = (sr1 - sr2) * np.sqrt(TRADING_DAYS)  # annualized delta
    return z, p, delta

CONTRASTS = [
    ("VMP(EW)",               "EW",               "VMP(EW) vs EW"),
    ("VMP(MSR(ledoit_wolf))", "MSR(ledoit_wolf)",  "VMP(MSR(LW)) vs MSR(LW)"),
    ("MSR(ledoit_wolf)",      "MSR(sample)",       "MSR(LW) vs MSR(sample)"),
    (None,                    "SWITCH(ledoit_wolf)", "SWITCH(v2a) vs SWITCH(LW)"),
    ("VMP(SWITCH(ledoit_wolf))", "VMP(MSR(ledoit_wolf))", "VMP(SWITCH(LW)) vs VMP(MSR(LW))"),
]

results_mem = []
for colA, colB, label in CONTRASTS:
    sA = switch_v2a if colA is None else strategy_returns(colA)
    sB = strategy_returns(colB)
    common = sA.index.intersection(sB.index)
    z, p, delta = memmel_test(sA.reindex(common), sB.reindex(common))
    sh_A = ann_sharpe(sA)
    sh_B = ann_sharpe(sB)
    results_mem.append({
        "Contrast": label,
        "Sharpe(A)": sh_A, "Sharpe(B)": sh_B,
        "ΔSharpe": delta, "z": z, "p-value": p,
        "Significant": "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns")),
    })

df_mem = pd.DataFrame(results_mem)
df_mem[["Sharpe(A)","Sharpe(B)","ΔSharpe","z"]] = df_mem[["Sharpe(A)","Sharpe(B)","ΔSharpe","z"]].round(3)
df_mem["p-value"] = df_mem["p-value"].apply(lambda x: f"{x:.4f}" if x >= 0.0001 else "<0.0001")
display(df_mem)
print("\nSignificance: *** p<0.001  ** p<0.01  * p<0.05  ns not significant")


> **Reading — §2.13:** The Memmel (2003) correction accounts for contemporaneous return correlation between paired portfolios — a standard t-test inflates significance when two strategies hold many of the same assets. Finding R1 (MSR(LW) vs. sample, p=0.005) survives; Finding R3 (SWITCH v2a, p=0.37) does not. Absence of significance is not evidence of no difference; the sign-test for VMP universality (p≈6×10⁻⁸) sidesteps the power problem via directional consistency across 24 tests. *See §4.1–4.3; Finding R1–R3.*

### 2.14 — BTC Survivorship Sensitivity

BTC-USD first traded on 2010-07-13, leaving 13.8% of the harness period (636 of 4,611 days) forward-filled at $0.06. Re-running eight comparable strategies on the 29-asset no-BTC universe quantifies the distortion; see Finding 15 in the paper.

In [ ]:
# Legacy comparison: 30-asset (with BTC, 2008-2026) vs 29-asset (without BTC, 2008-2026 subset)
base_legacy = pd.read_parquet('data/cache/portfolio_returns/24strategies_2008_2026.parquet')
vmp_legacy  = pd.read_parquet('data/cache/portfolio_returns/24strategies_vmp_2008_2026.parquet')
no_btc = pd.read_parquet('data/cache/portfolio_returns/8strategies_no_btc_2008_2026.parquet')
regime_sig_legacy = pd.read_parquet("data/cache/regime_signals.parquet")["dominant_regime"].dropna()
switch_v2a_legacy = assemble_switch_returns(
    base_legacy, regime_sig_legacy, SWITCH_V2A_RULE, "MDP(ledoit_wolf)"
).rename("SWITCH(v2a)")

COL_MAP = {
    'EW': 'EW', 'GMV(ledoit_wolf)': 'GMV(LW)', 'MSR(ledoit_wolf)': 'MSR(LW)',
    'MDP(ledoit_wolf)': 'MDP(LW)', 'HRP(ledoit_wolf)': 'HRP(LW)',
    'SWITCH(ledoit_wolf)': 'SWITCH(LW)', 'SWITCH(v2a)': 'SWITCH(v2a)',
    'VMP(MSR(ledoit_wolf))': 'VMP(MSR(LW))',
}

with_btc_s = {
    'EW': ann_sharpe(base_legacy['EW']),
    'GMV(ledoit_wolf)': ann_sharpe(base_legacy['GMV(ledoit_wolf)']),
    'MSR(ledoit_wolf)': ann_sharpe(base_legacy['MSR(ledoit_wolf)']),
    'MDP(ledoit_wolf)': ann_sharpe(base_legacy['MDP(ledoit_wolf)']),
    'HRP(ledoit_wolf)': ann_sharpe(base_legacy['HRP(ledoit_wolf)']),
    'SWITCH(ledoit_wolf)': ann_sharpe(base_legacy['SWITCH(ledoit_wolf)']),
    'SWITCH(v2a)': ann_sharpe(switch_v2a_legacy),
    'VMP(MSR(ledoit_wolf))': ann_sharpe(vmp_legacy['VMP(MSR(ledoit_wolf))']),
}
without_btc_s = {c: ann_sharpe(no_btc[c]) for c in no_btc.columns}
labels = [COL_MAP[k] for k in with_btc_s]
deltas = [with_btc_s[k] - without_btc_s[k] for k in with_btc_s]

x = np.arange(len(labels))
w = 0.35
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.bar(x - w/2, [with_btc_s[k] for k in with_btc_s], w, label='30-asset (with BTC)', color='#1f77b4')
ax.bar(x + w/2, [without_btc_s[k] for k in with_btc_s], w, label='29-asset (no BTC)', color='#ff7f0e')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel('Sharpe'); ax.set_title('Sharpe: 30-asset (with BTC) vs 29-asset (no BTC)', fontsize=10)
ax.legend(fontsize=8)

ax2 = axes[1]
colors_delta = ['#d62728' if d > 0 else '#1f77b4' for d in deltas]
ax2.bar(x, deltas, color=colors_delta)
ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=45, ha='right')
ax2.set_ylabel('Sharpe difference (with BTC - without BTC)')
ax2.set_title('BTC Impact on Sharpe (2008-2026, legacy 30-asset universe)', fontsize=10)
ax2.axhline(0, color='black', lw=0.8)

fig.tight_layout()
plt.show()

print(f"Median BTC impact: {np.median(deltas):.3f}")
print(f"Mean BTC impact:   {np.mean(deltas):.3f}")
print(f"All positive (BTC helps): {all(d > 0 for d in deltas)}")


> **Reading — §2.14:** Dropping BTC-USD from the 30-asset universe reduces median Sharpe by 0.229 across eight comparable strategies — well above the 0.02 materiality threshold. MSR(LW) and VMP(MSR(LW)) are most affected (+0.30 and +0.28), reflecting BTC's high-return contribution during trending episodes. HRP(LW) is the exception: Sharpe improves by 0.14 without BTC, because BTC's extreme volatility disrupts the correlation-cluster structure that HRP's dendrogram relies on. The headline findings — VMP universally improves, LW shrinkage helps MSR — hold in both universes. A clean no-forward-fill comparison would require starting from ~2014, shortening the sample to approximately 12 years.

### 2.15 — Asset-Class-Stratified Transaction Costs

The flat 10 bps assumption overstates costs for ETF-heavy strategies and understates for BTC-heavy ones. This section applies per-asset costs following §3.3.4 in the paper (Finding 13 update): 2 bps for fixed-income ETFs, 3 bps for broad/sector ETFs, 5 bps for single stocks and international ETFs, 30 bps for BTC.

In [ ]:
from aiam.evaluation.transaction_costs import apply_stratified_costs, STRATIFIED_COST_BPS

# Compute stratified net Sharpe for all 31 base + 31 VMP strategies
base31_ext = pd.read_parquet('data/cache/portfolio_returns/31strategies_29assets_2003_2026.parquet')
vmp31_ext  = pd.read_parquet('data/cache/portfolio_returns/31strategies_vmp_29assets_2003_2026.parquet')

def stratified_net(col, is_vmp=False):
    r = vmp31_ext[f'VMP({col})'] if is_vmp else base31_ext[col]
    w = weights_cache.get(col)
    return ann_sharpe(apply_stratified_costs(r, w)) if w is not None else np.nan

flat10, strat_ns, names_s, families_s = [], [], [], []
for col in base31_ext.columns:
    for is_v in [False, True]:
        disp = f'VMP({col})' if is_v else col
        # Use extended caches so constrained/LS strategies are covered
        _base_r = vmp31_ext[f'VMP({col})'] if is_v else base31_ext[col]
        _base_w = weights_cache.get(col)
        flat_n = ann_sharpe(apply_costs(_base_r, _base_w)) if _base_w is not None else np.nan
        strat_n = stratified_net(col, is_v)
        fam = FAMILY_MAP.get(col, 'EW (benchmark)')
        flat10.append(flat_n); strat_ns.append(strat_n)
        names_s.append(disp); families_s.append(fam)

fig, ax = plt.subplots(figsize=(9, 7))
lo, hi = 0.3, 1.55
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, zorder=0, label='45° line')
fam_colors = FAMILY_COLORS
for fam, col_c in fam_colors.items():
    idx = [i for i, f in enumerate(families_s) if f == fam]
    ax.scatter([flat10[i] for i in idx], [strat_ns[i] for i in idx],
               c=col_c, label=fam, alpha=0.7, s=40, zorder=3)
# Label BTC-heavy strategies
btc_labels = ['BL-Mom(LW)', 'MSR(LW)', 'EW', 'SWITCH(ledoit_wolf)']
for i, n in enumerate(names_s):
    disp_n = n.replace('ledoit_wolf', 'LW').replace('(oas)', '(OAS)')
    if disp_n in btc_labels and not n.startswith('VMP('):
        ax.annotate(disp_n, (flat10[i], strat_ns[i]), fontsize=7,
                    xytext=(5, 2), textcoords='offset points')
ax.set_xlabel('Net Sharpe (flat 10 bps)'); ax.set_ylabel('Net Sharpe (stratified)')
ax.set_title('Flat vs Stratified Transaction Costs — All 62 Strategy Records')
ax.legend(loc='upper left', fontsize=7); plt.tight_layout(); plt.show()

> **Reading — §2.15:** The scatter above the 45° line shows that stratified costs are cheaper than the flat 10 bps benchmark for almost all strategies, because fixed-income ETFs cost only 2 bps and broad equity ETFs only 3 bps. The largest beneficiaries are high-turnover equity strategies: FF3-Mom rises from 0.310 to 0.485 net Sharpe. BTC-heavy strategies (labeled) gain slightly from cheaper equity components; the 30 bps BTC cost matters only where BTC weight is large. The qualitative ranking — VMP regime and diversification strategies lead — survives unchanged under stratified costs.

---
## Part 3 — OOS Holdout Validation

Train period: 2003-01-02 – 2022-12-31 (20 years).  
Test period: 2023-01-01 – 2026-04-30 (3.3 years).  

The SWITCH(v2a) rule is re-derived from training-only regime-conditional Sharpe to avoid
data-snooping in regime assignment. All other strategies run continuously on the
walk-forward harness with no lookahead.


In [ ]:
from aiam.data.split import TRAIN_END, TEST_START

# Load pre-computed OOS series (output of build_switch_oos.py)
oos_df = pd.read_parquet("data/cache/portfolio_returns/switch_v2a_oos_29assets.parquet")

# Full 31-strategy returns for OOS comparison
train_sharpes, test_sharpes = {}, {}
for col in base31_all.columns:
    s = base31_all[col].dropna()
    train_s = s.loc[s.index <= TRAIN_END]
    test_s  = s.loc[s.index >= TEST_START]
    if len(train_s) > 50:
        train_sharpes[col] = ann_sharpe(train_s)
    if len(test_s) > 20:
        test_sharpes[col] = ann_sharpe(test_s)

# Add SWITCH OOS variants
for col in oos_df.columns:
    s = oos_df[col].dropna()
    tr = s.loc[s.index <= TRAIN_END]; te = s.loc[s.index >= TEST_START]
    if len(tr) > 50: train_sharpes[col] = ann_sharpe(tr)
    if len(te) > 20: test_sharpes[col] = ann_sharpe(te)

# Scatter: train Sharpe vs test Sharpe
common = sorted(set(train_sharpes) & set(test_sharpes))
tr_arr = np.array([train_sharpes[c] for c in common])
te_arr = np.array([test_sharpes[c] for c in common])

FAMILY_COLORS_EXT = {
    "EW (benchmark)": "#333333", "Classical MV": "#1f77b4", "Diversification": "#2ca02c",
    "Regime": "#9467bd", "TSMOM": "#ff7f0e", "Black-Litterman": "#d62728", "FF3 Factor": "#8c564b",
}
pt_colors = [FAMILY_COLORS_EXT.get(FAMILY_MAP.get(c, "Classical MV"), "#aaaaaa") for c in common]

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(tr_arr, te_arr, c=pt_colors, alpha=0.7, s=70, zorder=3)

lo = min(tr_arr.min(), te_arr.min()) - 0.1
hi = max(tr_arr.max(), te_arr.max()) + 0.1
ax.plot([lo, hi], [lo, hi], "k--", lw=0.8, alpha=0.4, label="Train = Test")
ax.axhline(0, color="grey", lw=0.6, alpha=0.4)
ax.axvline(0, color="grey", lw=0.6, alpha=0.4)
ax.set_xlabel("Train Sharpe (2003-2022)", fontsize=11)
ax.set_ylabel("Test Sharpe (2023-2026)", fontsize=11)
ax.set_title("OOS Holdout: Train vs Test Sharpe (29 assets)", fontsize=12, fontweight="bold")

top5 = sorted(zip(te_arr, tr_arr, common), reverse=True)[:5]
for te_v, tr_v, name in top5:
    label = name.replace("ledoit_wolf","LW").replace("(sample)","(s)").replace("_train_only","(OOS)")
    ax.annotate(label, (tr_v, te_v), textcoords="offset points", xytext=(6, 3), fontsize=7)

from matplotlib.patches import Patch
legend_items = [Patch(color=c, label=f) for f, c in FAMILY_COLORS_EXT.items()]
ax.legend(handles=legend_items, fontsize=8, loc="lower right")
ax.grid(True, alpha=0.2)
fig.tight_layout()
plt.show()

KEY = ["EW", "MSR(ledoit_wolf)", "MDP(ledoit_wolf)", "HRP(ledoit_wolf)",
       "SWITCH(ledoit_wolf)", "BL-Mom(LW)", "FF3-LowVol",
       "SWITCH_v2a_original", "SWITCH_v2a_train_only"]
print(f"\n{'Strategy':<35}  {'Train Sharpe':>12}  {'Test Sharpe':>12}")
print("-" * 65)
for c in KEY:
    tr = train_sharpes.get(c, float("nan"))
    te = test_sharpes.get(c, float("nan"))
    print(f"{c:<35}  {tr:>12.3f}  {te:>12.3f}")


> **Reading — §3:** The train-vs-test scatter tests whether full-sample Sharpe rankings persist
> out-of-sample. Points above the diagonal outperform in the test period relative to training;
> points below regress. The SWITCH(v2a) rule re-derived from training-only data is the cleanest
> OOS benchmark — any selection from the full-sample regime table constitutes hindsight bias.
> Strategies clustering above the diagonal in 2023–2026 demonstrate genuine post-rate-shock
> resilience; those collapsing below may have over-fit to the pre-2022 macro regime.
